# 12. Ders: Metin Benzerliği ve Algoritmik Eşleştirme (Buzdolabı Şefi - Bölüm 1)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Google'da "Makarnalı kolay tarifler" yazdığınızda, Google saniyeler içinde dünyadaki milyarlarca sayfa arasından sizin cümlenize 'en çok benzeyen' siteleri bulup getirir. Bunu nasıl yapar? Kelimeleri tek tek if-else ile eşleştirerek mi? Hayır, bu asırlar sürerdi.

Arama motorları ve tavsiye sistemleri (YouTube, Netflix önerileri dahil) metinleri ve özellikleri uzaydaki birer 'sayı dizisine' (Vektör) çevirir. Eğer iki vektör uzayda birbirine çok yakınsa, bu metinler birbirine benziyor demektir. GCA Coding Jam'in ünlü 'Buzdolabı Şefi' (Fridge Chef) projesinin ilk adımını atıyoruz. Kullanıcı buzdolabındaki malzemeleri (Örn: 'Domates, Yumurta, Biber') yazacak, biz de bu malzemeleri veritabanımızdaki (Pandas DataFrame) yüzlerce tarifle çarpıştırarak ona en uygun tarifi bulacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Vectorization (Vektörleştirme): Bir metni veya kelime grubunu (Örn: "Domates") matematiksel sayı dizilerine dönüştürme işlemidir.
- TF-IDF (Term Frequency - Inverse Document Frequency): Bir kelimenin belgede ne kadar sık geçtiğini, ancak genel dilde (veya tüm belgelerde) ne kadar nadir bulunduğunu oranlayan çok güçlü bir matematiksel formüldür. (Örn: 've' kelimesi her yerde vardır puanı düşüktür, 'Enginar' kelimesi nadirdir, puanı yüksektir).
- Cosine Similarity (Kosinüs Benzerliği): İki sayı dizisinin (vektörün) uzaydaki açılarına bakarak birbirlerine ne kadar benzediğini 0 ile 1 arasında (0: hiç benzemez, 1: tamamen aynı) hesaplayan bir geometri aracıdır.
- Dataset (Veri Seti): Tariflerimizin bulunduğu Pandas tablosu.

## Adım 3: Scikit-Learn ile TF-IDF Araçlarını Koda Dahil Etme

7. dersimizde öğrendiğimiz makine öğrenmesi kütüphanesi Scikit-Learn (sklearn), sadece tahmin yapmakla kalmaz, metinleri sayılara çevirmek ve benzerlik ölçmek için de harika araçlara sahiptir.

In [ ]:
import pandas as pd

# Kelimeleri matematiksel frekans dizisine çeviren vektörize aracı
from sklearn.feature_extraction.text import TfidfVectorizer

# İki dizi arasındaki matematiksel benzerliği ölçen geometri aracı
from sklearn.metrics.pairwise import cosine_similarity

print("Benzerlik motoru araçları başarıyla yüklendi.")

## Adım 4: Yemek Tarifleri Veri Setini (Dataset) Oluşturma

Buzdolabı şefinin bir zekası olması için önce bildiği tarifler olması gerekir. Küçük bir Pandas Veri Çerçevesi kuralım.

In [ ]:
# İçerisinde Yemek Adı ve Malzemeler olan küçük bir menü veritabanı
tarifler = {
    "Yemek Adı": ["Menemen", "Krep", "Tavuk Sote", "Makarna", "Salata"],
    "Malzemeler": [
        "domates biber yumurta yağ tuz",
        "süt un yumurta tuz yağ",
        "tavuk biber soğan domates yağ baharat",
        "makarna su tuz yağ salça",
        "marul domates salatalık limon zeytinyağı"
    ]
}

df_tarif = pd.DataFrame(tarifler)
print("Sisteme Yüklü Tarifler:")
print(df_tarif)

## Adım 5: Malzemeleri Vektörleştirme (TF-IDF)

Şimdi sistemdeki tüm malzemeleri kelime olmaktan çıkarıp devasa bir sayı matrisine (tablosuna) dönüştürüyoruz.

In [ ]:
# TfidfVectorizer aracını başlatıyoruz
vektor_donusturucu = TfidfVectorizer()

# .fit_transform() komutu hem kelimeleri öğrenir (fit) hem de onları sayılara dönüştürür (transform)
tarif_matrisi = vektor_donusturucu.fit_transform(df_tarif["Malzemeler"])

print("Sistem tüm malzemeleri okudu ve matematiksel olarak kodladı.")
# İpucu: tarif_matrisi artık harflerden değil, bilgisayarın anlayacağı 0.35, 0.67 gibi yüzlerce rakamdan oluşuyor.

## Adım 6: Kullanıcıdan Malzeme Girdisi Alma

Kullanıcının buzdolabında ne olduğunu öğrenelim. Bu girdiyi de makinenin anlaması için (tıpkı sistemdeki tarifler gibi) aynı vektörize aracından geçirip matematiksel matrise dönüştürmemiz ZORUNLUDUR. Yoksa elma ile armutu kıyaslamış oluruz.

In [ ]:
# Kullanıcının elindeki malzemeler
kullanici_malzemeleri = "yumurta ve süt var biraz da un"

# Kullanıcının girdisini aynı araçla dönüştürüyoruz (Dikkat: Sadece transform kullanıyoruz, 
# çünkü araç kelime dağarcığını (fit) tarif_matrisinden zaten öğrenmişti).
kullanici_matrisi = vektor_donusturucu.transform([kullanici_malzemeleri])

print("Kullanıcının buzdolabı verisi rakamlara dönüştürüldü.")

## Adım 7: Cosine Similarity (Kosinüs Benzerliği) ile Kıyaslama

Elimizde iki adet sayı yığını var: Birisi sistemdeki tüm tariflerin matrisi, diğeri kullanıcının buzdolabının matrisi. Şimdi bu ikisini uzayda birbirine çarpıştırıp aralarındaki benzerlik oranını bulacağız.

In [ ]:
# Benzerlik motorunu çalıştırıyoruz. 
# Birinci parametre kullanıcı, ikinci parametre sistemdeki tarifler.
benzerlik_skorlari = cosine_similarity(kullanici_matrisi, tarif_matrisi)

# Çıkan sonuçları daha rahat görmek için ilk sırayı alıyoruz
skorlar = benzerlik_skorlari[0]

print("Her bir tarif için bulunan benzerlik oranı (0 ile 1 arası):")
print(skorlar)

# Sonuçlara bakarsanız, 2. sıradaki (Krep) tarifinin skorunun çok yüksek olduğunu görebilirsiniz.

## Adım 8: En İyi Eşleşmeyi (Tarifi) Bulma ve Ekrana Basma

Skorlar arasında en büyük rakamı bulup, bunun hangi yemeğe denk geldiğini Pandas üzerinden kullanıcıya sunacağız.

In [ ]:
import numpy as np # Sayısal işlemler için kullanılan kütüphane

# argmax() fonksiyonu, bir liste içindeki en yüksek değerin indeksini (sıra numarasını) bulur.
en_iyi_indeks = np.argmax(skorlar)
en_iyi_skor = skorlar[en_iyi_indeks]

# Eğer hiçbir kelime eşleşmediyse (Skor 0 ise)
if en_iyi_skor == 0:
    print("Üzgünüm, buzdolabınızdaki malzemelerle yapılabilecek bir tarif bulamadım.")
else:
    onerilen_yemek = df_tarif["Yemek Adı"].iloc[en_iyi_indeks]
    gerekli_malzemeler = df_tarif["Malzemeler"].iloc[en_iyi_indeks]
    
    print("--- BUZDOLABI ŞEFİ ÖNERİSİ ---")
    print(f"Yapabileceğiniz En İyi Yemek: {onerilen_yemek}")
    print(f"(Sistem Eşleşme Güveni: Yüzde {en_iyi_skor*100:.0f})")
    print(f"Tarif İçin Gerekli Tam Malzemeler: {gerekli_malzemeler}")

## Adım 9: Dene ve Keşfet - Farklı Malzemeler

Kullanıcının sadece "domates, limon ve tuz" kelimelerini sisteme girdiğini düşünelim. Algoritmanın bu kelimeleri Menemen'e mi yoksa Salata'ya mı yönlendireceğini tahmin edin ve kodla test edin.

In [ ]:
test_malzeme = "Biraz domates var, biraz da limon ve tuz."

# 1. Girdiyi vektöre çevir (transform)
test_matrisi = vektor_donusturucu.transform([test_malzeme])

# 2. Cosine Similarity ile çarpıştır
test_skor = cosine_similarity(test_matrisi, tarif_matrisi)[0]

# Bütün tarifler için skoru yazdırarak sistemi denetleyin
for isim, skor in zip(df_tarif["Yemek Adı"], test_skor):
    print(f"{isim} eşleşme skoru: {skor:.2f}")

## Adım 10: Sıra Sende Görevi - Akıllı Kitap Tavsiye Motoru

Problem: Öğrendiğiniz eşleştirme algoritmasını (Cosine Similarity) kullanarak, kullanıcının sevdiği anahtar kelimelere göre ona kitap öneren bir kütüphane asistanı kodlamanız isteniyor.

Gereksinimler:
1. `kitaplar` adında bir Pandas DataFrame oluşturun. 
   - Sütun 1: "Kitap Adı" (Örn: Marslı, Sherlock Holmes, Yüzüklerin Efendisi)
   - Sütun 2: "Konular" (Örn: "uzay bilim kurgu macera gezegen", "dedektif gizem polisiye zeka", "büyü fantastik macera savaş")
2. `TfidfVectorizer` ile "Konular" sütununu matematiksel `kitap_matrisi`ne çevirin (`fit_transform`).
3. Kullanıcıdan `input()` kullanarak "Ne tür kitaplar seversin?" diye sorun.
4. Kullanıcının girdisini `transform` ile çevirip `cosine_similarity` ile karşılaştırın.
5. En yüksek skora sahip kitabın adını ekrana "Sizin için önerdiğimiz kitap: X" şeklinde yazdırın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 12. Dersi tamamlayarak dünyadaki milyar dolarlık arama motorlarının ve tavsiye algoritmalarının temelini oluşturan TF-IDF ve Vektör Uzay Modelini (Vector Space Model) kavradınız.

Bir sonraki dersten itibaren (Modül 4 - Kariyer Terzisi), çok daha uzun metinleri (makaleler, özgeçmişler) tek bir kelimeye değil, anlam bütünlüğüne (bağlama) göre analiz edip otomatik özet çıkaran sistemleri inceleyeceğiz.